# Paths and shapes

Beyond simple node-to-node connections, tikzfigure lets you draw arbitrary paths
from lists of `(x, y)` coordinates.

- drawing open and closed coordinate paths
- `filldraw` for filled closed shapes
- rounded corners
- per-segment styling with `segment_options`
- node paths and `.center` anchors for correct fills
- custom shapes (heart, star)
- parametric curves with NumPy (spiral, sun)
- `fill` for stroke-free fills and even-odd rule
- `clip` for clipping regions
- `path` for invisible named paths and mid-path labels

In [ ]:
import numpy as np

from tikzfigure import TikzFigure, arrows, styles, units

## Open and closed coordinate paths

Pass a list of `[x, y]` pairs (or `(x, y)` tuples) to `fig.draw()`. Set
`cycle=True` to close the path back to the starting point.

In [ ]:
fig = TikzFigure()

# Open path - a zigzag
zigzag = [[0, 0], [1, 1], [2, 0], [3, 1], [4, 0]]
fig.draw(zigzag, line_width=2, color="blue")

# Closed path - a triangle
triangle = [[6, 0], [8, 0], [7, 1.5]]
fig.draw(
    triangle,
    cycle=True,
    line_width=2,
    color="red",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Filled shapes with `filldraw`

`fig.filldraw()` is shorthand for a closed path that is both drawn (outline) and
filled. You can also use `fig.draw(..., fill="color")` for the same effect.

In [ ]:
fig = TikzFigure()

# Pentagon via filldraw
angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, 6)[:-1]
pentagon = list(
    zip(
        np.cos(angles) * 2 + 2,
        np.sin(angles) * 2 + 2,
    ),
)
fig.filldraw(
    pentagon,
    cycle=True,
    line_width=2,
    color="purple",
    fill="purple!20",
)

# Same via draw(..., fill=...)
hexagon_angles = np.linspace(0, 2 * np.pi, 7)[:-1]
hexagon = list(
    zip(
        np.cos(hexagon_angles) * 2 + 7,
        np.sin(hexagon_angles) * 2 + 2,
    ),
)
fig.draw(
    hexagon,
    cycle=True,
    line_width=2,
    color="teal",
    fill="teal!20",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Rounded corners

The `rounded_corners` parameter (in pt) smooths the corners of any path.

In [ ]:
fig = TikzFigure()

# Same square, with increasing corner radius
for i, radius in enumerate([0, 5, 15, 30]):
    x0 = i * 3.5
    square = [[x0, 0], [x0 + 2, 0], [x0 + 2, 2], [x0, 2]]
    fig.draw(
        square,
        cycle=True,
        line_width=2,
        color="black",
        fill="orange!20",
        rounded_corners=f"{radius}pt",
    )
    fig.node(
        x0 + 1,
        -0.5,
        content=f"{radius}pt",
        draw="none",
    )

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Per-segment styling

`segment_options` accepts a list of per-connector option dicts, one for each
`to` step in the path. Each dict uses keyword-style TikZ options (underscores
become spaces) plus an optional `"options"` key for flag-style items like
`styles.dashed` or `arrows.forward`.

Here four segments are given different colors and styles in a single `draw`
call:

In [ ]:
fig = TikzFigure()

waypoints = [[0, 0], [2, 2], [4, 0], [6, 2], [8, 0]]

fig.draw(
    waypoints,
    line_width=2,
    segment_options=[
        {"color": "red"},
        {"color": "blue", "options": [styles.dashed], "line_width": 1.5},
        {"color": "green!60!black", "line_width": 3},
        {"color": "purple", "options": [styles.dotted], "line_width": 2},
    ],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

Per-segment bending works the same way. Here three nodes are connected with
arrows where each segment curves in a different direction:

In [ ]:
fig = TikzFigure()

specs = [("src", 0, 1, "teal"), ("mid", 4, 3, "orange"), ("dst", 8, 1, "purple")]
for label, x, y, color in specs:
    fig.node(
        (x, y),
        label=label,
        shape="circle",
        fill=f"{color}!25",
        draw=f"{color}!70",
        minimum_size=1.0 * units.cm,
        content=label.upper(),
    )

fig.draw(
    ["src.center", "mid.center", "dst.center"],
    arrows=arrows.forward,
    options=[styles.thick],
    segment_options=[
        {"bend_left": 30},  # src → mid curves upward-left
        {"bend_right": 25},  # mid → dst curves downward-right
    ],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Fluent node chaining

If you're connecting named nodes, `.to(...)` is a concise alternative to
building `segment_options` by hand. Options passed to `.to(...)` affect only
that connector.

In [ ]:
fig = TikzFigure()

A = fig.node(
    (0, 1),
    label="A",
    shape="circle",
    fill="teal!25",
    draw="teal!70",
    minimum_size=1.0 * units.cm,
    content="A",
)
B = fig.node(
    (4, 3),
    label="B",
    shape="circle",
    fill="orange!25",
    draw="orange!70",
    minimum_size="1cm",
    content="B",
)
C = fig.node(
    (8, 1),
    label="C",
    shape="circle",
    fill="purple!25",
    draw="purple!70",
    minimum_size="1cm",
    content="C",
)

fig.draw(
    A.to(B).to(C, options=styles.bend_left()),
    arrows=arrows.forward,
    options=[styles.thick],
    color="teal!70",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

Arc segments fit into the same fluent API. Unlike `.to(...)`, `.arc(...)` does
not consume another waypoint — it appends a TikZ `arc[...]` operation at the
current point, and you can keep chaining afterward:

In [ ]:
fig = TikzFigure()

A = fig.node(
    (0, 0),
    label="A",
    shape="circle",
    fill="teal!25",
    draw="teal!70",
    minimum_size="1cm",
    content="A",
)
B = fig.node(
    (4, 0),
    label="B",
    shape="circle",
    fill="orange!25",
    draw="orange!70",
    minimum_size="1cm",
    content="B",
)
C = fig.node(
    (7, 1.5),
    label="C",
    shape="circle",
    fill="purple!25",
    draw="purple!70",
    minimum_size="1cm",
    content="C",
)

fig.draw(
    A.to(B)
    .arc(start_angle=-90, end_angle=90, radius="1.5cm")
    .to(C, options=styles.bend_right()),
    arrows=arrows.forward,
    options=[styles.thick],
    color="blue!70",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

Use `"out"` and `"in"` keys for explicit departure and arrival angles (in
degrees). This gives precise control over how each segment enters and leaves its
endpoints — useful when `bend_left`/`bend_right` doesn't give you the exact
shape you need:

In [ ]:
fig = TikzFigure()

for label, x, y, color in [
    ("A", 0, 2, "blue"),
    ("B", 5, 4, "orange"),
    ("C", 10, 2, "purple"),
]:
    fig.node(
        (x, y),
        label=label,
        shape="circle",
        fill=f"{color}!20",
        draw=f"{color}!70",
        minimum_size="1.2cm",
        content=f"${label}$",
    )

fig.draw(
    ["A.center", "B.center", "C.center"],
    arrows=arrows.forward,
    options=[styles.thick],
    segment_options=[
        {"out": 45, "in": 195},  # A→B: leave at 45°, arrive at 195°
        {"out": -15, "in": 135},  # B→C: leave at -15°, arrive at 135°
    ],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

`looseness` scales how far the control handles extend, making the same bend
either tighter (`looseness < 1`) or more exaggerated (`looseness > 1`). Here the
two segments of the same symmetric path use opposite looseness values:

In [ ]:
fig = TikzFigure()

for label, x, y in [("A", 0, 2), ("B", 5, 2), ("C", 10, 2)]:
    fig.node(
        (x, y),
        label=label,
        shape="circle",
        fill="gray!15",
        draw="gray!60",
        minimum_size="1cm",
        content=f"${label}$",
    )

fig.draw(
    ["A.center", "B.center", "C.center"],
    arrows=arrows.forward,
    options=[styles.thick],
    segment_options=[
        {"bend_left": 50, "looseness": 2.5},  # A→B: wide exaggerated arc
        {"bend_right": 50, "looseness": 0.5},  # B→C: tight compressed arc
    ],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

All segment options can be combined freely. Here each segment gets its own
color, line style, and curve geometry in a single `draw` call:

In [ ]:
fig = TikzFigure()

for label, x, y, color in [
    ("A", 0, 0, "teal"),
    ("B", 4, 3, "red"),
    ("C", 8, 0, "blue"),
]:
    fig.node(
        (x, y),
        label=label,
        shape="circle",
        fill=f"{color}!20",
        draw=f"{color}!60",
        minimum_size="1.2cm",
        content=f"${label}$",
    )

fig.draw(
    ["A.center", "B.center", "C.center"],
    segment_options=[
        {  # A→B
            "options": [arrows.forward],
            "color": "teal!80",
            "line_width": 2,
            "out": 60,
            "in": 240,
        },
        {  # B→C
            "options": [arrows.forward, styles.dashed],
            "color": "red!70",
            "line_width": 1.5,
            "out": -60,
            "in": 120,
        },
    ],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Node paths and `.center` anchors

When you fill a closed path that goes through named nodes, TikZ routes to each
node's *border* by default. For circular nodes this means the filled area has
indented corners — it doesn't look like a triangle.

Appending `.center` to a node label in the path list routes the segment through
the node's geometric center instead. The path then bounds a proper filled
polygon.

In [ ]:
fig = TikzFigure()

node_specs = [
    ("top", 3, 4, "blue", r"$v_1$"),
    ("bl", 0, 0, "blue", r"$v_2$"),
    ("br", 6, 0, "blue", r"$v_3$"),
]
for label, x, y, color, text in node_specs:
    fig.node(
        (x, y),
        label=label,
        shape="circle",
        draw=f"{color}!60",
        fill="white",
        minimum_size="1.4cm",
        content=text,
    )

# Border anchors: the fill "leaks" under the node circles
fig.draw(
    ["top", "bl", "br"],
    cycle=True,
    fill="red!25",
    draw="red!60",
    line_width=1,
    fill_opacity=0.6,
)

# .center anchors: fills the exact triangle spanned by node centres
fig.draw(
    ["top.center", "bl.center", "br.center"],
    cycle=True,
    fill="blue!30",
    draw="blue!70",
    line_width=1.5,
    fill_opacity=0.7,
)

# Labels outside the shapes
fig.node(
    (7.5, 2),
    content=r"\textcolor{red!60}{border anchors}",
    draw="none",
    anchor="west",
    font=r"\small",
)
fig.node(
    (7.5, 1),
    content=r"\textcolor{blue!70}{.center anchors}",
    draw="none",
    anchor="west",
    font=r"\small",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

You can mix anchors: only the nodes that need a specific anchor require the dot
suffix. Any TikZ anchor string works — `"north"`, `"south east"`, `"west"`, etc.

In [ ]:
fig = TikzFigure()

fig.node(
    (0, 0),
    label="A",
    shape="rectangle",
    draw="black",
    fill="gray!15",
    minimum_width="2cm",
    minimum_height="1cm",
    content="A",
)
fig.node(
    (5, 2),
    label="B",
    shape="rectangle",
    draw="black",
    fill="gray!15",
    minimum_width="2cm",
    minimum_height="1cm",
    content="B",
)
fig.node(
    (10, 0),
    label="C",
    shape="rectangle",
    draw="black",
    fill="gray!15",
    minimum_width="2cm",
    minimum_height="1cm",
    content="C",
)

# Connect from the east side of A, enter B from the west, leave from east, arrive at C.west
fig.draw(
    ["A.east", "B.west"],
    arrows=arrows.forward,
    options=[styles.thick],
    color="teal",
    segment_options=[{"bend_left": 15}],
)
fig.draw(
    ["B.east", "C.west"],
    arrows=arrows.forward,
    options=[styles.thick],
    color="orange",
    segment_options=[{"bend_right": 15}],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Heart shape

A heart with circular lobes drawn by `fig.arc()` and a pointed bottom built from
a chained path.

In [ ]:
width, height = 1.75, 2.0

fig = TikzFigure()

fig.colorlet("lightred", "red!40!white")

A = fig.node(-width, height)
B = fig.node((0, 0))
C = fig.node(width, height)
D = fig.node(0, height)

fig.draw(
    A.to(B, options=["out=-90, in=135"])
    .to(C, options=["out=45, in=-90"])
    .to(D, options=["in=80, out=100"])
    .to(A, options=["in=80, out=100"]),
    color="red",
    line_width=4,
    cycle=True,
    center=True,
    fill="lightred",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Star shape

A five-pointed star drawn from six points (the last one closes back to the
first).

In [ ]:
fig = TikzFigure()

star = [[0, 0], [1, 2.5], [2, 0], [-0.5, 1.5], [2.5, 1.5], [0, 0]]

fig.draw(
    star,
    line_width=3,
    cycle=True,
    color="Gold",
    fill="yellow!50!white",
)

fig.show()
print(fig)

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Parametric curves with NumPy

Any iterable of `(x, y)` pairs works as a path, so NumPy arrays plug in
naturally for smooth parametric curves.

In [ ]:
fig = TikzFigure()

# Spiral
N = 300
theta = np.linspace(0, 8 * np.pi, N)
r = np.linspace(0.1, 3, N)
spiral = list(
    zip(r * np.cos(theta) + 4, r * np.sin(theta) + 4),
)
fig.draw(spiral, line_width=2, color="teal")

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Sun: circle core + radial rays

A filled circle for the core, plus short line segments for rays at equal angles.

In [ ]:
fig = TikzFigure()

cx, cy, R = 4, 4, 1.0  # center and core radius

# Core
theta = np.linspace(0, 2 * np.pi, 60)
core = list(
    zip(
        cx + R * np.cos(theta),
        cy + R * np.sin(theta),
    ),
)
fig.filldraw(
    core,
    cycle=True,
    line_width=2,
    color="orange",
    fill="yellow!60",
)

# Rays
for angle in np.linspace(
    0,
    2 * np.pi,
    12,
    endpoint=False,
):
    x0 = cx + R * np.cos(angle)
    y0 = cy + R * np.sin(angle)
    x1 = cx + (R + 0.7) * np.cos(angle)
    y1 = cy + (R + 0.7) * np.sin(angle)
    fig.draw(
        [(x0, y0), (x1, y1)],
        line_width=2,
        color="orange",
    )

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Fill without stroke

`fig.fill()` fills a closed path without drawing any outline — the equivalent of
`\fill[...]` in TikZ. Use it when you want pure flat colour with no border.

In [ ]:
fig = TikzFigure()

# Three overlapping filled triangles — no outlines at all
triangles = [
    ([[1, 0], [4, 0], [2.5, 2.5]], "red!60"),
    ([[2, 0], [5, 0], [3.5, 2.5]], "blue!50"),
    ([[3, 0], [6, 0], [4.5, 2.5]], "green!60"),
]
for pts, color in triangles:
    fig.fill(
        pts,
        cycle=True,
        fill=color,
        fill_opacity=0.6,
    )

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

The `even_odd_rule` flag corrects the fill for self-intersecting paths. Without
it, TikZ fills the interior using the non-zero winding rule, which fills the
overlapping centre region twice — visually it looks like a solid shape. With
`even_odd_rule=True` the centre hole is punched out correctly.

In [ ]:
fig = TikzFigure()

# A five-pointed star is self-intersecting
angles = np.linspace(np.pi / 2, np.pi / 2 + 4 * np.pi, 6)[:-1]
star_pts = list(
    zip(
        np.cos(angles) * 2 + 2,
        np.sin(angles) * 2 + 2,
    ),
)

# Default winding rule: centre is filled solid
fig.fill(
    star_pts,
    cycle=True,
    fill="orange!80",
    fill_opacity=0.9,
)
fig.node(
    (2, -0.3),
    content="winding rule (default)",
    draw="none",
    font=r"\small",
    anchor="center",
)

# Even-odd rule: centre is hollow
angles2 = np.linspace(np.pi / 2, np.pi / 2 + 4 * np.pi, 6)[:-1]
star_pts2 = list(
    zip(
        np.cos(angles2) * 2 + 8,
        np.sin(angles2) * 2 + 2,
    ),
)
fig.fill(
    star_pts2,
    cycle=True,
    fill="teal!70",
    fill_opacity=0.9,
    even_odd_rule=True,
)
fig.node(
    (8, -0.3),
    content="even odd rule",
    draw="none",
    font=r"\small",
    anchor="center",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Clip

`fig.clip()` installs a clipping path — all subsequent drawing is masked to the
given boundary. Wrap the clip together with the clipped content inside a
`\begin{scope}...\end{scope}` via `fig.raw()` so the clip doesn't leak into the
rest of the figure.

In [ ]:
fig = TikzFigure()

# Draw a full circle for reference (unclipped)
fig.draw(
    list(
        zip(
            np.cos(np.linspace(0, 2 * np.pi, 60)) * 2 + 2,
            np.sin(np.linspace(0, 2 * np.pi, 60)) * 2 + 2,
        ),
    ),
    cycle=True,
    color="gray!40",
    line_width=1,
)

# Open a scope, clip to a rectangle, then draw a dense grid inside
fig.raw(r"\begin{scope}")
clip_rect = [[0.5, 0.5], [3.5, 0.5], [3.5, 3.5], [0.5, 3.5]]
fig.clip(clip_rect, cycle=True)

for i in np.linspace(0, 4, 20):
    fig.draw(
        [(0, i), (4, i)],
        color="blue!40",
        line_width=0.4,
    )
    fig.draw(
        [(i, 0), (i, 4)],
        color="blue!40",
        line_width=0.4,
    )

fig.raw(r"\end{scope}")

# Anything drawn after the scope is unclipped again
fig.node(
    (2, -0.5),
    content="clipped grid inside circle",
    draw="none",
    font=r"\small",
    anchor="center",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## Invisible path

`fig.path()` traces a path without drawing or filling anything. It has two
common uses:

**Named paths** — register a path under a name so the `intersections` TikZ
library can compute where two paths cross.

**Mid-path labels** — place a text node at a specific position along an
invisible connector using `segment_options` with a `"node"` key.

In [ ]:
fig = TikzFigure()

# Three nodes
for label, x, y, color in [
    ("A", 0, 0, "teal"),
    ("B", 5, 3, "orange"),
    ("C", 10, 0, "purple"),
]:
    fig.node(
        (x, y),
        label=label,
        shape="circle",
        fill=f"{color}!20",
        draw=f"{color}!70",
        minimum_size="1.2cm",
        content=f"${label}$",
    )

# Visible arrows
fig.draw(
    ["A.center", "B.center"],
    arrows=arrows.forward,
    options=[styles.thick],
    color="teal!70",
    segment_options=[{"bend_left": 20}],
)
fig.draw(
    ["B.center", "C.center"],
    arrows=arrows.forward,
    options=[styles.thick],
    color="orange!80",
    segment_options=[{"bend_right": 20}],
)

# Invisible paths with mid-path labels
fig.path(
    ["A.center", "B.center"],
    segment_options=[
        {
            "node": {
                "content": r"$e_1$",
                "above": True,
                "sloped": True,
                "font": r"\small",
            },
            "bend_left": 20,
        },
    ],
)
fig.path(
    ["B.center", "C.center"],
    segment_options=[
        {
            "node": {
                "content": r"$e_2$",
                "above": True,
                "sloped": True,
                "font": r"\small",
            },
            "bend_right": 20,
        },
    ],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())